# Reproducing results:

## How does our system compare to existing error detection approaches?

Instead of using the datasets from the original GitHub repository, I selected a different dataset that has previously been used with the Raha architecture. For the rest of the implementation, I will use the Raha library and attempt to reproduce the results by following the code provided in the repository’s benchmark script.

In [1]:
import os
os.environ["PYTHONUTF8"] = "1"
import sys
import time
import shutil
from pathlib import Path
import numpy
import prettytable
import matplotlib.pyplot
import pandas as pd
import sklearn.linear_model

# Raha is an older project with outdated dependencies, so some compatibility adjustments
# have been made to ensure proper execution of the experiment.
pd.Series.iteritems = pd.Series.items
pd.DataFrame.iteritems = pd.DataFrame.items
pd.DataFrame.applymap = pd.DataFrame.map

# Scikit-learn compatibility
import sklearn.linear_model

OriginalSGDClassifier = sklearn.linear_model.SGDClassifier

def CompatibleSGDClassifier(*args, **kwargs):
    if kwargs.get("loss") == "log":
        kwargs["loss"] = "log_loss"
    return OriginalSGDClassifier(*args, **kwargs)

sklearn.linear_model.SGDClassifier = CompatibleSGDClassifier

import raha

In [ ]:
ITERATIONS = 1
standalone_baseline = ["dBoost", "NADEEF", "KATARA", "ActiveClean", "Raha"]
#standalone_baseline = ["dBoost", "NADEEF", "KATARA", "Raha"]
aggregator_baseline = ["Min-k", "Maximum Entropy", "Metadata Driven", "Raha"]
tuple_baseline = ["ActiveClean", "Raha"]

In [3]:

DATASETS_DIR = Path("datasets").resolve()

DATASETS = sorted([
    folder.name
    for folder in DATASETS_DIR.iterdir()
    if folder.is_dir()
    and (folder / "dirty.csv").exists()
    and (folder / "clean.csv").exists()
])

def get_dataset_dictionary(dataset_name):
    folder = DATASETS_DIR / dataset_name
    return {
        "name": dataset_name,
        "path": str(folder / "dirty.csv"),
        "clean_path": str(folder / "clean.csv")
    }

print("Datasets found:", DATASETS)


Datasets found: ['beers', 'flights', 'hospital', 'movies_1', 'rayyan', 'tax', 'toy']


In [4]:
results_standalone = {
    system: {dataset: [] for dataset in DATASETS}
    for system in standalone_baseline
}

for r in range(ITERATIONS):
    detector = raha.detection.Detection()   
    detector.VERBOSE = False
    competitor = raha.baselines.Baselines()
    
    for dataset_name in DATASETS:
        dataset_dictionary = get_dataset_dictionary(dataset_name)
        d = raha.dataset.Dataset(dataset_dictionary)
        for system in standalone_baseline[::-1]:
            if system == "dBoost":
                detections = competitor.run_dboost(dataset_dictionary)
            elif system == "NADEEF":
                detections = competitor.run_nadeef(dataset_dictionary)
            elif system == "KATARA":
                detections = competitor.run_katara(dataset_dictionary)
            elif system == "ActiveClean":
                detections = competitor.run_activeclean(dataset_dictionary)
            elif system == "Raha":
                detections = detector.run(dataset_dictionary)

            evaluation = d.get_data_cleaning_evaluation(detections)[:3]
            results_standalone[system][dataset_name].append(evaluation)
            print(system + " on " + dataset_name)

I just load strategies' results as they have already been run on the dataset!


Raha on beers
KATARA on beers
NADEEF on beers
dBoost on beers


I just load strategies' results as they have already been run on the dataset!


Raha on flights
KATARA on flights
NADEEF on flights
dBoost on flights


I just load strategies' results as they have already been run on the dataset!


Raha on hospital
KATARA on hospital
NADEEF on hospital
dBoost on hospital


I just load strategies' results as they have already been run on the dataset!


Raha on movies_1
KATARA on movies_1
NADEEF on movies_1
dBoost on movies_1


PermissionError: [Errno 13] Het proces heeft geen toegang tot het bestand omdat het door een ander
proces wordt gebruikt: 'C:\\Users\\lcsmi\\AppData\\Local\\Temp\\rayyan-1182302093955441369916165107723418544313042437617.csv'

In [ ]:
results_tuple = {
    system: {dataset: [] for dataset in DATASETS}
    for system in tuple_baseline
}

for r in range(ITERATIONS):
    detector = raha.detection.Detection()
    detector.VERBOSE = False
    competitor = raha.baselines.Baselines()

    for dataset_name in DATASETS:
        dataset_dictionary = get_dataset_dictionary(dataset_name)
        d = raha.dataset.Dataset(dataset_dictionary)

        for system in tuple_baseline[::-1]:
            if system == "ActiveClean":
                detections = competitor.run_activeclean(dataset_dictionary)
            elif system == "Raha":
                detections = detector.run(dataset_dictionary)

            evaluation = raha.utilities.get_tuple_wise_evaluation(d, detections)
            results_tuple[system][dataset_name].append(evaluation)

In [ ]:
sampling_range = [20, 40, 60, 80, 100]

results_aggregators = {
    system: {
        dataset: {s: [] for s in sampling_range}
        for dataset in DATASETS
    }
    for system in aggregator_baseline
}

for r in range(ITERATIONS):
    detector = raha.detection.Detection()
    detector.VERBOSE = False
    competitor = raha.baselines.Baselines()

    for dataset_name in DATASETS:
        dataset_dictionary = get_dataset_dictionary(dataset_name)
        d = raha.dataset.Dataset(dataset_dictionary)

        for s in sampling_range:
            for system in aggregator_baseline[::-1]:
                if system == "Min-k":
                    detections = competitor.run_min_k(dataset_dictionary)
                elif system == "Maximum Entropy":
                    detections = competitor.run_maximum_entropy(dataset_dictionary, s)
                elif system == "Metadata Driven":
                    detections = competitor.run_metadata_driven(dataset_dictionary, s)
                elif system == "Raha":
                    detector.LABELING_BUDGET = s
                    detections = detector.run(dataset_dictionary)

                evaluation = d.get_data_cleaning_evaluation(detections)[:3]
                results_aggregators[system][dataset_name][s].append(evaluation)

## Results

In [ ]:

table_1 = prettytable.PrettyTable(["Approach"] + DATASETS)

for system in standalone_baseline:
    row = [system]

    for dataset_name in DATASETS:
        p, r, f = numpy.mean(
            numpy.array(results_standalone[system][dataset_name]),
            axis=0
        )
        row.append(f"{p:.2f}, {r:.2f}, {f:.2f}")

    table_1.add_row(row)

print("Stand-alone error detection (Precision, Recall, F1)")
print(table_1)


In [ ]:

table_2 = prettytable.PrettyTable(["Approach"] + DATASETS)

for system in tuple_baseline:
    row = [system]

    for dataset_name in DATASETS:
        p, r, f = numpy.mean(
            numpy.array(results_tuple[system][dataset_name]),
            axis=0
        )
        row.append(f"{p:.2f}, {r:.2f}, {f:.2f}")

    table_2.add_row(row)

print("Tuple-wise error detection (Precision, Recall, F1)")
print(table_2)


In [ ]:

table_3 = prettytable.PrettyTable(["Approach"] + DATASETS)

f_scores = {}

for system in aggregator_baseline:
    row = [system]

    for dataset_name in DATASETS:
        f_list = [
            numpy.mean(
                numpy.array(results_aggregators[system][dataset_name][s]),
                axis=0
            )[2]
            for s in sampling_range
        ]

        row.append(", ".join(f"{f:.2f}" for f in f_list))
        f_scores[(system, dataset_name)] = f_list

    table_3.add_row(row)

print(f"Aggregator F1 scores for labeled tuples: {sampling_range}")
print(table_3)


In [ ]:

fig, axs = matplotlib.pyplot.subplots(
    nrows=1,
    ncols=len(DATASETS),
    figsize=(5 * len(DATASETS), 5),
    squeeze=False
)

for i, ax in enumerate(axs.flat):
    ax.set_title(DATASETS[i])
    ax.set_xlabel("Labeled Tuples Count")
    ax.set_ylabel("F1 Score")
    ax.set_ylim([0.0, 1.0])
    ax.grid(True)

    for system in aggregator_baseline:
        f_list = f_scores[(system, DATASETS[i])]

        ax.plot(
            [0] + sampling_range,
            [0 if system != "Min-k" else f_list[0]] + f_list,
            label=system
        )

    ax.legend()

fig.suptitle("Comparison with the error detection aggregators")
matplotlib.pyplot.tight_layout()
matplotlib.pyplot.show()
